In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nandand14/botsv3-enriched-features-behavioral/botsv3_enriched_features_behavioral.csv


In [2]:

import pandas as pd
import numpy as np

# ── Load ──────────────────────────────────────────────────────────────────────
csv_path = "/kaggle/input/datasets/nandand14/botsv3-enriched-features-behavioral/botsv3_enriched_features_behavioral.csv"
df = pd.read_csv(csv_path)

# ── Basic inspection ──────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION 1 – SHAPE & OVERVIEW")
print("=" * 70)
print(f"Shape        : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>3}. {col}")

print("\nData types:")
print(df.dtypes.to_string())

print("\nFirst 5 rows:")
print(df.head().to_string())

# ── Required columns check ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 2 – REQUIRED COLUMNS PRESENCE CHECK")
print("=" * 70)

KEY_COLS   = ["user_name", "cloudtrail_time", "is_anomaly"]
FEAT_COLS  = [
    "hour", "day_of_week", "is_business_hours", "is_weekend",
    "time_since_last_change", "change_velocity", "change_magnitude",
    "rarity_score", "user_activity", "is_wildcard", "blast_radius",
    "privilege_direction", "is_error", "action_frequency",
    "resource_centrality", "neighbor_count", "clustering_coeff",
    "pagerank_score", "betweenness",
]
ALL_REQUIRED = KEY_COLS + FEAT_COLS

present   = [c for c in ALL_REQUIRED if c in df.columns]
missing   = [c for c in ALL_REQUIRED if c not in df.columns]

print(f"  Required columns  : {len(ALL_REQUIRED)}")
print(f"  ✅  Present        : {len(present)}")
print(f"  ❌  Missing        : {len(missing)}")
if missing:
    for m in missing:
        print(f"       – {m}")
else:
    print("  All 22 required columns found!")

# ── Missing value audit ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 3 – MISSING VALUES IN CRITICAL COLUMNS")
print("=" * 70)

null_series = {col: df[col].isnull().sum() for col in ALL_REQUIRED if col in df.columns}
null_report = pd.DataFrame([
    {"column": col, "missing_count": cnt, "missing_%": round(cnt / len(df) * 100, 3)}
    for col, cnt in null_series.items() if cnt > 0
])

if null_report.empty:
    print("  ✅  Zero missing values across all 22 required columns.")
else:
    print(f"  ⚠️  Columns with missing values ({len(null_report)}):")
    print(null_report.to_string(index=False))

# ── Unique users & rows-per-user ───────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 4 – USER-LEVEL STATISTICS")
print("=" * 70)

# Work with rows that have a valid user_name
df_named = df[df["user_name"].notna()].copy()
rows_per_user = df_named["user_name"].value_counts().sort_values(ascending=False)
n_users = rows_per_user.shape[0]
n_null_users = df["user_name"].isnull().sum()

print(f"  Total rows              : {df.shape[0]:,}")
print(f"  Rows with null user_name: {n_null_users:,}  ({n_null_users/len(df)*100:.1f}%)")
print(f"  Rows with valid user    : {len(df_named):,}")
print(f"  Unique named users      : {n_users:,}")
print(f"  Avg rows/user           : {rows_per_user.mean():.1f}")
print(f"  Min rows/user           : {rows_per_user.min()}")
print(f"  Max rows/user           : {rows_per_user.max():,}")
print(f"\n  Rows per user breakdown:")
print(rows_per_user.to_string())

# Users with < 10 rows
sparse_users = rows_per_user[rows_per_user < 10]
print(f"\n  Users with < 10 rows : {len(sparse_users):,}")
if len(sparse_users) > 0:
    print("  (These may be unsuitable for sliding-window modelling)")
    print(sparse_users.to_string())

# ── Feature dtype sanity ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 5 – FEATURE COLUMN DTYPE SUMMARY")
print("=" * 70)
existing_feats = [c for c in FEAT_COLS if c in df.columns]
print(df[existing_feats].dtypes.to_string())

# ── Sliding window verdict ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SECTION 6 – SLIDING WINDOW VALIDITY VERDICT")
print("=" * 70)

all_cols_present  = len(missing) == 0
# user_name nulls are structural (service accounts / AWS internal); feature cols are clean
feat_nulls        = {c: df[c].isnull().sum() for c in FEAT_COLS if c in df.columns and df[c].isnull().sum() > 0}
no_feature_nulls  = len(feat_nulls) == 0
enough_data_users = (rows_per_user >= 10).sum()
pct_valid_users   = enough_data_users / n_users * 100 if n_users > 0 else 0
majority_valid    = pct_valid_users >= 50

print(f"  All 22 required columns present        : {'YES ✅' if all_cols_present else 'NO ❌'}")
print(f"  user_name null rows (likely svc accts) : {n_null_users:,} ({n_null_users/len(df)*100:.1f}%)")
print(f"  No missing values in feature columns   : {'YES ✅' if no_feature_nulls else 'NO ❌'  + str(feat_nulls)}")
print(f"  Unique named users                     : {n_users:,}")
print(f"  Users with ≥10 rows                    : {enough_data_users:,} / {n_users:,}  ({pct_valid_users:.0f}%)")
print(f"  Majority of users valid for windows    : {'YES ✅' if majority_valid else 'NO ❌'}")

# Named-user data is valid if cols present, features clean, majority of users have enough rows
verdict = all_cols_present and no_feature_nulls and majority_valid
print("\n" + "─" * 70)
print(f"  ➜  DATA VALID FOR SLIDING WINDOWS?  →  {'✅  YES' if verdict else '❌  NO'}")
if not verdict:
    if n_null_users > 0:
        print(f"     NOTE: {n_null_users:,} rows have null user_name (service/system accounts).")
        print(f"     ➜  After filtering to named users, the dataset IS valid.")
        print(f"     ➜  RECOMMENDATION: filter rows where user_name is not null before windowing.")
print("─" * 70)


SECTION 1 – SHAPE & OVERVIEW
Shape        : 6,571 rows  x  41 columns

All columns (41):
    1. hour
    2. day_of_week
    3. is_business_hours
    4. is_weekend
    5. time_since_last_change
    6. change_velocity
    7. change_magnitude
    8. rarity_score
    9. user_activity
   10. is_wildcard
   11. blast_radius
   12. privilege_direction
   13. is_error
   14. action_frequency
   15. resource_centrality
   16. neighbor_count
   17. clustering_coeff
   18. pagerank_score
   19. betweenness
   20. error_rate_recent
   21. cloudtrail_time
   22. event_name
   23. user_name
   24. source_ip
   25. net_bytes_total
   26. net_bytes_out
   27. net_unique_ips
   28. net_suspicious_ports
   29. net_exfil_score
   30. net_connections
   31. end_unique_processes
   32. end_suspicious_processes
   33. end_privileged_events
   34. end_logon_events
   35. end_process_creation
   36. is_anomaly
   37. anomaly_score
   38. triggered_rules
   39. confidence
   40. severity
   41. rule_count

Dat

In [3]:

import pandas as pd
import numpy as np
import json

# ── Constants (re-declared so this block is self-contained) ───────────────────
FEAT_COLS = [
    "hour", "day_of_week", "is_business_hours", "is_weekend",
    "time_since_last_change", "change_velocity", "change_magnitude",
    "rarity_score", "user_activity", "is_wildcard", "blast_radius",
    "privilege_direction", "is_error", "action_frequency",
    "resource_centrality", "neighbor_count", "clustering_coeff",
    "pagerank_score", "betweenness",
]
WINDOW_SIZE = 10

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 – CLEAN THE DATAFRAME
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 1 – DATA CLEANING")
print("=" * 70)

_clean = df.copy()

# Parse cloudtrail_time as datetime (UTC, ignore errors → NaT then drop)
_clean["cloudtrail_time"] = pd.to_datetime(_clean["cloudtrail_time"], utc=True, errors="coerce")
_n_nat = _clean["cloudtrail_time"].isna().sum()
if _n_nat > 0:
    print(f"  ⚠️  {_n_nat} rows with unparseable cloudtrail_time dropped.")
    _clean = _clean.dropna(subset=["cloudtrail_time"])
else:
    print("  ✅  cloudtrail_time parsed – zero NaT values.")

# Label column: session_label ← is_anomaly
_clean["session_label"] = _clean["is_anomaly"].astype(np.int32)

# Ensure username column exists (group key required by time-based windowing)
if "username" in _clean.columns:
    _n_filled = _clean["username"].isna().sum()
    _clean["username"] = _clean["username"].fillna("unknown_service")
    print(f"  ✅  username: {_n_filled:,} nulls filled with 'unknown_service'.")
else:
    # Derive username from user_name (the canonical column in this CSV)
    _n_filled = _clean["user_name"].isna().sum()
    _clean["username"] = _clean["user_name"].fillna("unknown_service")
    print(f"  ✅  username derived from user_name: {_n_filled:,} nulls filled with 'unknown_service'.")

# Sort globally by cloudtrail_time
_clean = _clean.sort_values("cloudtrail_time").reset_index(drop=True)

# Drop full duplicates
_n_before_dedup = len(_clean)
_clean = _clean.drop_duplicates().reset_index(drop=True)
_n_dropped = _n_before_dedup - len(_clean)
print(f"  ✅  Duplicates dropped: {_n_dropped:,} rows removed.")
print(f"  ✅  Clean dataframe shape: {_clean.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 – TIME-BASED SLIDING WINDOWS PER USER
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 2 – BUILDING TIME-BASED SLIDING WINDOWS")
print("=" * 70)

_WINDOW_DURATION = np.timedelta64(10, "m")   # 10-minute time window

X_list          = []            # each entry: ndarray (WINDOW_SIZE, len(FEAT_COLS))
y_list          = []            # each entry: int (0 or 1)
event_name_list = []            # each entry: list[str] of event_names in the window
user_window_counts = {}         # {username: n_windows}

_users = _clean["username"].unique()
_skipped_users = []

for _user in _users:
    # Group by username, sort by timestamp
    _grp = (
        _clean[_clean["username"] == _user]
        .sort_values("cloudtrail_time")
        .reset_index(drop=True)
    )
    _n_rows = len(_grp)

    _features = _grp[FEAT_COLS].values.astype(np.float32)          # (n_rows, 19)
    _labels   = _grp["session_label"].values.astype(np.int32)       # (n_rows,)
    _times    = _grp["cloudtrail_time"].values                       # (n_rows,) datetime64

    _user_windows = 0

    # For each event anchor, compute a time-based window [window_start, window_end)
    for _i in range(_n_rows):
        _window_start = _times[_i]
        _window_end   = _window_start + _WINDOW_DURATION

        # Collect ALL events in [window_start, window_end) via boolean mask
        _mask        = (_times >= _window_start) & (_times < _window_end)
        _win_feats   = _features[_mask]      # (k, 19) – k varies per window
        _win_labels  = _labels[_mask]        # (k,)

        _k = len(_win_feats)
        if _k == 0:
            continue

        # EDIT 1: Label via majority vote (mean >= 0.5) instead of np.max
        _win_label = int(np.mean(_win_labels) >= 0.5)

        # Pad with np.zeros or truncate to exactly WINDOW_SIZE → (WINDOW_SIZE, 19)
        if _k < WINDOW_SIZE:
            _pad = np.zeros((WINDOW_SIZE - _k, len(FEAT_COLS)), dtype=np.float32)
            _win_feats_fixed = np.vstack([_win_feats, _pad])
        else:
            _win_feats_fixed = _win_feats[:WINDOW_SIZE]

        X_list.append(_win_feats_fixed)    # (10, 19)
        y_list.append(_win_label)          # scalar int

        # EDIT 2: Track event_names for this window (up to WINDOW_SIZE events)
        event_name_list.append(_grp["event_name"].values[_mask][:WINDOW_SIZE].tolist())

        _user_windows += 1

    if _user_windows == 0:
        _skipped_users.append((_user, _n_rows))
    else:
        user_window_counts[_user] = _user_windows

if _skipped_users:
    print(f"  ⚠️  Users skipped (0 windows produced):")
    for _u, _r in _skipped_users:
        print(f"       {_u!r} → {_r} rows")
else:
    print(f"  ✅  All users produced at least one window – no users skipped.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 – ASSEMBLE ARRAYS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 3 – ASSEMBLING ARRAYS")
print("=" * 70)

X_sequences = np.array(X_list, dtype=np.float32)   # (N, 10, 19)
y_labels     = np.array(y_list, dtype=np.int32)     # (N,)

# Shape assertions
assert X_sequences.ndim == 3,                   f"Expected 3-D X, got {X_sequences.ndim}-D"
assert X_sequences.shape[1] == WINDOW_SIZE,     f"Expected {WINDOW_SIZE} timesteps, got {X_sequences.shape[1]}"
assert X_sequences.shape[2] == len(FEAT_COLS),  f"Expected {len(FEAT_COLS)} features, got {X_sequences.shape[2]}"
assert y_labels.ndim == 1,                      f"Expected 1-D y, got {y_labels.ndim}-D"
assert len(X_sequences) == len(y_labels),       "X and y length mismatch"
assert len(event_name_list) == len(y_labels),   "event_name_list length mismatch"

print(f"  X shape  : {X_sequences.shape}   (windows × timesteps × features)")
print(f"  y shape  : {y_labels.shape}      (windows,)")
print(f"  event_name_list length: {len(event_name_list):,}")

# Class breakdown
_n_anomaly = int((y_labels == 1).sum())
_n_normal  = int((y_labels == 0).sum())
_total_w   = len(y_labels)
_ratio     = _n_anomaly / _n_normal if _n_normal > 0 else float("inf")

print(f"\n  Total windows  : {_total_w:,}")
print(f"  Anomaly (1)    : {_n_anomaly:,}  ({_n_anomaly/_total_w*100:.2f}%)")
print(f"  Normal  (0)    : {_n_normal:,}  ({_n_normal/_total_w*100:.2f}%)")
print(f"  Anomaly ratio  : {_ratio:.4f}  ({_ratio*100:.2f}% of normal)")

# Per-user window contribution (time-based grouping)
print(f"\n  Per-user window contribution (time-based grouping):")
for _u, _cnt in sorted(user_window_counts.items(), key=lambda x: -x[1]):
    print(f"    {_u!r:<35} → {_cnt:,} windows")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 – SAVE FILES
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 4 – SAVING FILES")
print("=" * 70)

np.save("X_sequences.npy", X_sequences)
np.save("y_labels.npy",    y_labels)
print("  ✅  X_sequences.npy saved.")
print("  ✅  y_labels.npy saved.")

# EDIT 2 (continued): Save event_name_list as JSON
with open("event_names_per_window.json", "w") as _f:
    json.dump(event_name_list, _f)
print("  ✅  event_names_per_window.json saved.")

# Determine class imbalance severity and LSTM readiness
_class_imbalance_ok = _ratio < 0.5
_enough_windows     = _total_w >= 100
_lstm_ready         = _total_w > 0 and X_sequences.shape[2] == len(FEAT_COLS)

if _ratio < 0.05:
    _imbalance_label = "SEVERE (consider oversampling / class weights)"
elif _ratio < 0.20:
    _imbalance_label = "MODERATE (class weights recommended)"
elif _ratio < 0.50:
    _imbalance_label = "MILD"
else:
    _imbalance_label = "BALANCED"

_verdict_str = "✅  READY FOR LSTM TRAINING" if _lstm_ready else "❌  NOT READY"

# Write summary report
_summary_lines = [
    "=" * 70,
    "SLIDING WINDOW SUMMARY REPORT",
    "=" * 70,
    "",
    "── DATA CLEANING ───────────────────────────────────────────────────────",
    f"  cloudtrail_time parsed as datetime UTC : YES",
    f"  NaT rows dropped                       : {_n_nat}",
    f"  username nulls filled ('unknown_svc')  : {_n_filled}",
    f"  Duplicate rows dropped                 : {_n_dropped}",
    f"  Clean dataframe shape                  : {_clean.shape}",
    "",
    "── WINDOW CONSTRUCTION ─────────────────────────────────────────────────",
    f"  Window size                            : {WINDOW_SIZE} timesteps",
    f"  Window duration                        : 10 minutes (time-based)",
    f"  Feature columns                        : {len(FEAT_COLS)}",
    f"  Label column                           : session_label (= is_anomaly)",
    f"  Labelling strategy                     : int(np.mean(session_label) >= 0.5) – majority vote across 10-min window",
    f"  Users processed                        : {len(user_window_counts)}",
    f"  Users skipped (0 windows)              : {len(_skipped_users)}",
    "",
    "── ARRAY SHAPES ────────────────────────────────────────────────────────",
    f"  X_sequences : {X_sequences.shape}",
    f"  y_labels    : {y_labels.shape}",
    "",
    "── CLASS DISTRIBUTION ──────────────────────────────────────────────────",
    f"  Total windows  : {_total_w:,}",
    f"  Anomaly (1)    : {_n_anomaly:,}  ({_n_anomaly/_total_w*100:.2f}%)",
    f"  Normal  (0)    : {_n_normal:,}  ({_n_normal/_total_w*100:.2f}%)",
    f"  Anomaly/Normal : {_ratio:.4f}",
    f"  Imbalance      : {_imbalance_label}",
    "",
    "── PER-USER WINDOW CONTRIBUTION (TIME-BASED) ───────────────────────────",
]
for _u, _cnt in sorted(user_window_counts.items(), key=lambda x: -x[1]):
    _summary_lines.append(f"  {_u:<40} : {_cnt:,} windows")

_summary_lines += [
    "",
    "── FILES SAVED ─────────────────────────────────────────────────────────",
    "  X_sequences.npy            (float32 array)",
    "  y_labels.npy               (int32 array)",
    "  event_names_per_window.json (list of event name lists per window)",
    "  window_summary.txt          (this report)",
    "",
    "── LSTM READINESS VERDICT ──────────────────────────────────────────────",
    f"  {_verdict_str}",
    "=" * 70,
]

_summary_text = "\n".join(_summary_lines)

with open("window_summary.txt", "w") as _f:
    _f.write(_summary_text)
print("  ✅  window_summary.txt saved.")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 – FINAL REPORT
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(f"  Data clean status     : ✅  {_clean.shape[0]:,} rows × {_clean.shape[1]} cols")
print(f"  Total windows created : {_total_w:,}")
print(f"  X_sequences shape     : {X_sequences.shape}  ✅  (N, {WINDOW_SIZE}, {len(FEAT_COLS)})")
print(f"  y_labels shape        : {y_labels.shape}  ✅  (N,)")
print(f"  event_name_list       : {len(event_name_list):,} entries  ✅")
print(f"  Anomaly (1)           : {_n_anomaly:,}  ({_n_anomaly/_total_w*100:.2f}%)")
print(f"  Normal  (0)           : {_n_normal:,}  ({_n_normal/_total_w*100:.2f}%)")
print(f"  Class imbalance       : {_imbalance_label}  (anomaly/normal = {_ratio:.4f})")
print(f"  Labelling strategy    : majority vote (mean >= 0.5)")
print(f"  LSTM readiness verdict: {_verdict_str}")
print("=" * 70)


STEP 1 – DATA CLEANING
  ✅  cloudtrail_time parsed – zero NaT values.
  ✅  username derived from user_name: 1,146 nulls filled with 'unknown_service'.
  ✅  Duplicates dropped: 2,987 rows removed.
  ✅  Clean dataframe shape: (3584, 43)

STEP 2 – BUILDING TIME-BASED SLIDING WINDOWS
  ✅  All users produced at least one window – no users skipped.

STEP 3 – ASSEMBLING ARRAYS
  X shape  : (3584, 10, 19)   (windows × timesteps × features)
  y shape  : (3584,)      (windows,)
  event_name_list length: 3,584

  Total windows  : 3,584
  Anomaly (1)    : 847  (23.63%)
  Normal  (0)    : 2,737  (76.37%)
  Anomaly ratio  : 0.3095  (30.95% of normal)

  Per-user window contribution (time-based grouping):
    'splunk_access'                     → 1,503 windows
    'unknown_service'                   → 953 windows
    'web_admin'                         → 567 windows
    'bstoll'                            → 492 windows
    'btun'                              → 69 windows

STEP 4 – SAVING FILES
  ✅  X

In [4]:

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ─────────────────────────────────────────────
# 0.  Pure-numpy helpers (no sklearn needed)
# ─────────────────────────────────────────────

def stratified_split(X, y, val_ratio=0.20, test_ratio=0.20, seed=42):
    """Stratified 60/20/20 split using pure numpy."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    train_idx, val_idx, test_idx = [], [], []
    for c in classes:
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_test = max(1, int(np.round(n * test_ratio)))
        n_val  = max(1, int(np.round(n * val_ratio)))
        test_idx.extend(idx[:n_test])
        val_idx.extend(idx[n_test:n_test + n_val])
        train_idx.extend(idx[n_test + n_val:])
    return (np.array(train_idx), np.array(val_idx), np.array(test_idx))


def compute_f1(y_true, y_pred_bin):
    tp = int(((y_pred_bin == 1) & (y_true == 1)).sum())
    fp = int(((y_pred_bin == 1) & (y_true == 0)).sum())
    fn = int(((y_pred_bin == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0, prec, rec


def compute_auc_roc(y_true, y_scores):
    """Efficient trapezoidal AUC-ROC via numpy sort (no sklearn)."""
    order = np.argsort(-y_scores)           # descending probability order
    y_sorted = y_true[order]
    pos = y_true.sum(); neg = len(y_true) - pos
    tpr, fpr, cum_tp, cum_fp = [0.0], [0.0], 0, 0
    for label in y_sorted:
        if label == 1: cum_tp += 1
        else:          cum_fp += 1
        tpr.append(cum_tp / pos if pos > 0 else 0.0)
        fpr.append(cum_fp / neg if neg > 0 else 0.0)
    tpr.append(1.0); fpr.append(1.0)
    return float(np.trapz(tpr, fpr))


def confusion_matrix_np(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return np.array([[tn, fp], [fn, tp]])


# ─────────────────────────────────────────────
# 1. Load Data
# ─────────────────────────────────────────────
X_sequences = np.load("X_sequences.npy").astype(np.float32)  # (N, 10, 19)
y_labels     = np.load("y_labels.npy").astype(np.float32)     # (N,)

print(f"Loaded  X: {X_sequences.shape},  y: {y_labels.shape}")
print(f"Class distribution — Anomaly: {y_labels.mean()*100:.1f}%  |  Normal: {(1-y_labels).mean()*100:.1f}%")

# ─────────────────────────────────────────────
# 2. Stratified 60 / 20 / 20 Split
# ─────────────────────────────────────────────
train_idx, val_idx, test_idx = stratified_split(X_sequences, y_labels, val_ratio=0.20, test_ratio=0.20)

X_train, y_train = X_sequences[train_idx], y_labels[train_idx]
X_val,   y_val   = X_sequences[val_idx],   y_labels[val_idx]
X_test,  y_test  = X_sequences[test_idx],  y_labels[test_idx]

print(f"\nSplit sizes — Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
for split_name, y_s in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"  {split_name}: anomaly={y_s.mean()*100:.1f}%  normal={(1-y_s).mean()*100:.1f}%")

# ─────────────────────────────────────────────
# 3. DataLoaders
# ─────────────────────────────────────────────
def make_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

# ─────────────────────────────────────────────
# 4. LSTM Model
# ─────────────────────────────────────────────
class LSTMAnomalyDetector(nn.Module):
    def __init__(self, input_size=19, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc      = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)           # (batch, seq_len, hidden)
        last_hidden  = lstm_out[:, -1, :]    # last time step
        out          = self.fc(last_hidden)  # (batch, 1)
        return self.sigmoid(out).squeeze(1)  # (batch,) → P_seq per sample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

lstm_model = LSTMAnomalyDetector(input_size=19, hidden_size=64, num_layers=2, dropout=0.3).to(device)
print(lstm_model)
total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# ─────────────────────────────────────────────
# 5. Class-Weighted BCELoss
#    pos_weight_val derived from actual class distribution in windowing output:
#      Anomaly (1): 23.63%  →  anomaly_pct = 23.63
#      Normal  (0): 76.37%  →  normal_pct  = 76.37
#      pos_weight_val = normal_pct / anomaly_pct
# ─────────────────────────────────────────────
_n_total    = len(y_labels)
_n_anomaly  = int((y_labels == 1).sum())
_n_normal   = int((y_labels == 0).sum())
anomaly_pct = _n_anomaly / _n_total * 100.0   # e.g. 23.63
normal_pct  = _n_normal  / _n_total * 100.0   # e.g. 76.37

pos_weight_val = normal_pct / anomaly_pct
print(f"\nClass distribution from loaded y_labels:")
print(f"  Anomaly (1): {_n_anomaly:,}  ({anomaly_pct:.2f}%)")
print(f"  Normal  (0): {_n_normal:,}  ({normal_pct:.2f}%)")
print(f"  Computed pos_weight_val = {normal_pct:.2f} / {anomaly_pct:.2f} = {pos_weight_val:.4f}")

bce_base = nn.BCELoss(reduction="none")

def weighted_bce(preds, targets, pos_weight=pos_weight_val):
    losses  = bce_base(preds, targets)
    weights = torch.where(targets == 1,
                          torch.full_like(targets, pos_weight),
                          torch.ones_like(targets))
    return (losses * weights).mean()

optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

# ─────────────────────────────────────────────
# 6. Training + Validation Loop
# ─────────────────────────────────────────────
def evaluate_loader(loader):
    lstm_model.eval()
    all_preds, all_targets = [], []
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = lstm_model(xb)
            loss  = weighted_bce(preds, yb)
            total_loss += loss.item() * len(yb)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())
    avg_loss    = total_loss / len(loader.dataset)
    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)
    bin_preds   = (all_preds >= 0.5).astype(int)
    f1_val, _, _  = compute_f1(all_targets, bin_preds)
    auc_val       = compute_auc_roc(all_targets, all_preds)
    return avg_loss, f1_val, auc_val

EPOCHS = 30
print(f"\n{'='*70}")
print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>9}  {'Val F1':>7}  {'Val AUC':>8}")
print(f"{'─'*70}")

lstm_train_history = []

for epoch in range(1, EPOCHS + 1):
    lstm_model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = lstm_model(xb)
        loss  = weighted_bce(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)

    train_loss_epoch          = epoch_loss / len(train_loader.dataset)
    val_loss, val_f1, val_auc = evaluate_loader(val_loader)

    lstm_train_history.append({
        "epoch": epoch, "train_loss": train_loss_epoch,
        "val_loss": val_loss, "val_f1": val_f1, "val_auc": val_auc
    })
    print(f"{epoch:>6d}  {train_loss_epoch:>11.5f}  {val_loss:>9.5f}  {val_f1:>7.4f}  {val_auc:>8.4f}")

print(f"{'='*70}")

# ─────────────────────────────────────────────
# 7. Final Test Set Evaluation
# ─────────────────────────────────────────────
lstm_model.eval()
test_preds_list, test_targets_list = [], []
test_total_loss = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = lstm_model(xb)
        loss  = weighted_bce(preds, yb)
        test_total_loss += loss.item() * len(yb)
        test_preds_list.extend(preds.cpu().numpy())
        test_targets_list.extend(yb.cpu().numpy())

lstm_test_loss    = test_total_loss / len(test_loader.dataset)
test_preds_arr    = np.array(test_preds_list)
test_targets_arr  = np.array(test_targets_list)
test_bin_preds    = (test_preds_arr >= 0.5).astype(int)

lstm_test_f1, test_prec, test_rec = compute_f1(test_targets_arr, test_bin_preds)
lstm_test_auc = compute_auc_roc(test_targets_arr, test_preds_arr)
lstm_test_cm  = confusion_matrix_np(test_targets_arr, test_bin_preds)

# Classification report
tn, fp, fn, tp = lstm_test_cm[0,0], lstm_test_cm[0,1], lstm_test_cm[1,0], lstm_test_cm[1,1]
n_pos  = int(test_targets_arr.sum())
n_neg  = int((1 - test_targets_arr).sum())
acc    = (tp + tn) / len(test_targets_arr)

print(f"\n{'━'*70}")
print("  FINAL TEST SET RESULTS")
print(f"{'━'*70}")
print(f"  Test Loss    : {lstm_test_loss:.5f}")
print(f"  Test F1      : {lstm_test_f1:.4f}")
print(f"  Test AUC-ROC : {lstm_test_auc:.4f}")
print(f"  Test Accuracy: {acc:.4f}")
print(f"  Precision    : {test_prec:.4f}  |  Recall: {test_rec:.4f}")

print(f"\nConfusion Matrix (rows=actual, cols=predicted):")
print(f"                  Pred Normal  Pred Anomaly")
print(f"  Actual Normal     {tn:>7}       {fp:>7}")
print(f"  Actual Anomaly    {fn:>7}       {tp:>7}")

print(f"\nClassification Report:")
print(f"{'─'*55}")
print(f"  {'Class':<14} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Support':>9}")
prec_n = tn / (tn + fn) if (tn + fn) > 0 else 0.0
rec_n  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1_n   = 2*prec_n*rec_n/(prec_n+rec_n) if (prec_n+rec_n) > 0 else 0.0
print(f"  {'Normal':<14} {prec_n:>7.4f} {rec_n:>7.4f} {f1_n:>7.4f} {n_neg:>9}")
print(f"  {'Anomaly':<14} {test_prec:>7.4f} {test_rec:>7.4f} {lstm_test_f1:>7.4f} {n_pos:>9}")
print(f"{'─'*55}")
print(f"  {'Accuracy':<14} {'':>7} {'':>7} {acc:>7.4f} {len(test_targets_arr):>9}")
print(f"{'━'*70}")

# ─────────────────────────────────────────────
# 8. Save Model
# ─────────────────────────────────────────────
torch.save(lstm_model.state_dict(), "lstm_model.pt")
print("\n✅  lstm_model.pt saved successfully.")


Loaded  X: (3584, 10, 19),  y: (3584,)
Class distribution — Anomaly: 23.6%  |  Normal: 76.4%

Split sizes — Train: 2152 | Val: 716 | Test: 716
  Train: anomaly=23.7%  normal=76.3%
  Val: anomaly=23.6%  normal=76.4%
  Test: anomaly=23.6%  normal=76.4%

Using device: cpu
LSTMAnomalyDetector(
  (lstm): LSTM(19, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Trainable parameters: 55,105

Class distribution from loaded y_labels:
  Anomaly (1): 847  (23.63%)
  Normal  (0): 2,737  (76.37%)
  Computed pos_weight_val = 76.37 / 23.63 = 3.2314

 Epoch   Train Loss   Val Loss   Val F1   Val AUC
──────────────────────────────────────────────────────────────────────


/tmp/ipykernel_17/1562168355.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(tpr, fpr))


     1      0.78203    0.34544   0.8882    0.9785
     2      0.30289    0.20139   0.9169    0.9940
     3      0.21385    0.17531   0.9169    0.9945
     4      0.17963    0.16965   0.9440    0.9947
     5      0.16884    0.12140   0.9257    0.9972
     6      0.15798    0.11588   0.9375    0.9975
     7      0.14528    0.10483   0.9357    0.9977
     8      0.13356    0.11080   0.9388    0.9973
     9      0.12875    0.10933   0.9302    0.9967
    10      0.11299    0.09099   0.9337    0.9975
    11      0.11089    0.12912   0.9467    0.9973
    12      0.12199    0.09364   0.9441    0.9975
    13      0.11340    0.11888   0.9467    0.9977
    14      0.10514    0.08866   0.9499    0.9977
    15      0.11108    0.09475   0.9467    0.9976
    16      0.12449    0.09108   0.9499    0.9978
    17      0.13896    0.11974   0.9471    0.9972
    18      0.12503    0.12082   0.9443    0.9970
    19      0.10756    0.09227   0.9467    0.9977
    20      0.09345    0.10594   0.9275    0.9967
